# 第14回：模擬Kaggle改善会

**今日の問い：限られた時間で、次に何を試すか。**

上から順に実行してください。`TRY`は全員、`CHANGE`は値を1つ変える練習、
`CHALLENGE`は余裕がある人向けです。分からないコードは、セル全体ではなく
気になる数行をM365 Copilotへ貼って相談します。


In [ ]:
from pathlib import Path

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.tomlがある勉強会フォルダ内で実行してください")

ROOT = find_repo_root()
DATA = ROOT / "data"
print("教材フォルダ:", ROOT)


## この回でできるようになること

        - 限られた時間で実験を優先順位付けする
- 特徴量・モデル・閾値を分離して評価する
- 誤分類を群別に調べて改善仮説を作る

        ### 進み方

        `CORE`は同期90分で扱う本線、`DEEP DIVE`は時間があれば扱う深掘り、
        `SELF-STUDY`は任意自習です。すべて終わらなくても次回へ進めます。

        ### 先に押さえる言葉

        - Leaderboard overfitting：順位表へ過度に合わせること
- 誤分類分析：外した試料の共通点を調べる作業
- 閾値調整：確率からクラスへの境界を変えること
- 実験統合：有効な変更を再検証しながら組み合わせること

        > **実行前の30秒予想**：今日の問いに、今の言葉で仮の答えを書いてから始めます。


In [ ]:
import pandas as pd
train=pd.read_csv(DATA / "local_competition" / "train.csv")
test=pd.read_csv(DATA / "local_competition" / "test.csv")
answers=pd.read_csv(DATA / "local_competition" / "instructor_answers.csv")


## 5人の担当

1. 欠損補完
2. 特徴量（最適温度からの距離）
3. モデルの深さ
4. 判定閾値
5. 誤分類の確認

全員が同じ`random_state=42`とF1を使い、担当箇所以外は変えません。


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

improved_train=train.copy()
improved_test=test.copy()
for frame in [improved_train, improved_test]:
    frame["temperature_distance"] = abs(frame["temperature_c"] - 78)
target="active"
ignored=["sample_id", "experiment_date", "smiles", target]
features=[c for c in improved_train.columns if c not in ignored]
numeric=improved_train[features].select_dtypes(include="number").columns.tolist()
categorical=[c for c in features if c not in numeric]
preprocess=ColumnTransformer([
    ("数値", SimpleImputer(strategy="median"), numeric),
    ("カテゴリ", Pipeline([("補完", SimpleImputer(strategy="most_frequent")), ("one_hot", OneHotEncoder(handle_unknown="ignore"))]), categorical),
])
model=Pipeline([("前処理", preprocess), ("モデル", RandomForestClassifier(n_estimators=300, max_depth=3, class_weight="balanced", random_state=42))])
X_train, X_valid, y_train, y_valid=train_test_split(improved_train[features], improved_train[target], test_size=0.25, random_state=42, stratify=improved_train[target])
model.fit(X_train, y_train)
print("改善案のローカルF1:", round(f1_score(y_valid, model.predict(X_valid)), 3))


In [ ]:
model.fit(improved_train[features], improved_train[target])
improved_submission=pd.DataFrame({"sample_id": improved_test["sample_id"], "active": model.predict(improved_test[features])})
output=ROOT / "workspace" / "submission_improved.csv"
improved_submission.to_csv(output, index=False)
local_score=f1_score(answers["active"], answers.merge(improved_submission, on="sample_id", suffixes=("_true", "_pred"))["active_pred"])
print("模擬Leaderboard F1:", round(local_score, 3))
print("保存先:", output)


## 実験ログ

改善しても悪化しても、`変更点 / ローカルF1 / 模擬Leaderboard F1 / 気づき`を1行で記録します。Leaderboardだけ改善し、ローカル検証が悪化した案は慎重に扱います。


## DEEP DIVE：結果を一段深く読む

        次のセルは、数値を出して終わらず「どの条件で、なぜそう見えるか」を調べる発展です。

        ### 出力を見る観点

        - ローカル改善とLeaderboard改善の一致を確認する
- 改善幅が偶然でないか再分割で見る
- 誤分類群にデータ不足や分布差がないか調べる


In [ ]:
validation_result = X_valid[["scaffold_group"]].copy()
validation_result["正解"] = y_valid
validation_result["予測"] = model.predict(X_valid)
validation_result["誤分類"] = validation_result["正解"] != validation_result["予測"]
error_by_group = validation_result.groupby("scaffold_group").agg(件数=("誤分類", "size"), 誤分類数=("誤分類", "sum"), 誤分類率=("誤分類", "mean"))
display(error_by_group.sort_values(["誤分類率", "件数"], ascending=False).round(3))


## よくある誤り

        - 5人の変更を一度に統合する
- Leaderboardだけを目的関数にする
- 検証データで選んだ閾値を同じデータで報告する

        ## SELF-STUDY（任意・30〜60分）

        - 系列別の件数・F1・誤分類数を表にする
- 最終案をゼロから再実行して同じ提出を作る

        成果は完成したコードでなくても、予想・変更点・出力・解釈を4行で残せば十分です。

        ## 振り返りチェック

        1. 次の実験を何で優先するか
2. ローカルとLeaderboardがずれたら何を疑うか
3. 改善を統合する順序はどうするか

        答えに詰まった項目が、次に見返す場所です。暗記ではなくNotebookの該当セルを指せればOKです。
